### Coleta de dados de Temperatura

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos:
<pre>
- Temperatura   -> variável 2m_temperature, retorna a temperatura em Kelvin, será necessário uma conversão (subtrair -273,15)
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil




In [1]:
import cdsapi
import sys, os
import xarray as xr
import dask.dataframe as dd
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [2]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)

c:\Marco Conti\Projetos\mais_einstein\Ondas_Calor


In [3]:
def get_cdsapi_authentication():
    url = os.getenv("ECMWF_DATASTORES_URL")
    key = os.getenv("ECMWF_DATASTORES_KEY")
    return url, key

def get_t2m(year):
    dataset = "derived-era5-single-levels-daily-statistics"
    request = {
        "product_type": "reanalysis",
        "variable": ["2m_temperature"],
        "year": f"{year}",
        "month": [
            "01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"
        ],
        "day": [
            "01", "02", "03", "04", "05", "06", "07", "08", "09",
            "10", "11", "12", "13", "14", "15", "16", "17", "18",
            "19", "20", "21", "22", "23", "24", "25", "26", "27",
            "28", "29", "30", "31"
        ],
        "daily_statistic": "daily_mean",
        "time_zone": "utc-03:00",
        "frequency": "1_hourly",
        # Retangulo geográfico definido por Norte, Oeste, Sul e Leste em graus onde está o Brasil
        "area": [6      # Norte
                ,-74    # Oeste
                ,-34    # Sul
                ,-38]   # Leste          
    }

    # Informações de autenticação estão em:
    # C:\Users\DRT90628\.ecmwfdatastoresrc
    # *** Criar um novo contrato de autenticação deverá ser criado usando um usuário de serviços do Einstein
    url, key = get_cdsapi_authentication()

    client = \
        cdsapi.Client(url = url
                     ,key = key
        )

    ret_download = client.retrieve(dataset, request).download()

    os.rename(ret_download, f"{DATA_PATH_ROOT}\ERA5-temperaturas\ERA5_t2m_{year}.nc")

    return ret_download

def convert_t2m_dataset_to_spark_dataframe(project_path, ret_download ):

    with xr.open_dataset(f"{project_path}\{ret_download}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:

        # Transforma o Dataset em um Spark Dataframe
        df_dask        = ds.to_dask_dataframe()
        df_dask_c      = df_dask.compute()
        df_temperatura = spark.createDataFrame(df_dask_c)

    return df_temperatura
    

def transform_data(df_temperatura):
    drop_cols = ["valid_time", "t2m", "number"]

    df_temperatura_final = \
        (df_temperatura
            .withColumns({"data_medicao"    : F.col("valid_time").cast("date")
                         ,"indicador"       : F.lit("temperatura") 
                         ,"valor"           : (F.col("t2m") - F.lit(273.15)).cast("double") # Converte a temperatura de Kelvin para Celsius
                         ,"unidade_medida"  : F.lit("celsius")})
            .drop(*drop_cols)
    )

    df_temperatura_final = \
        (df_temperatura_final
            .select("data_medicao"
                   ,"latitude"
                   ,"longitude"
                   ,"indicador"
                   ,"valor"
                   ,"unidade_medida"))

    return df_temperatura_final

def write_data(df_temperatura_final, write_path, ano):
    # df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)

    df_temperatura_final.toPandas().to_parquet(f"{write_path}\\ERA5-temperaturas\\{ano}\\ERA5_temperatura.parquet")

def remove_aux_file(file_name):
    os.remove(file_name)


In [4]:

# # Converte os dados de temperatura para um Spark Dataframe
# ret_download = "ea1aa17d3130f15e9827810d611337cb.nc"
# df_temperatura       = convert_t2m_dataset_to_spark_dataframe(PROJECT_PATH, ret_download)

# # Converte a temperatura de Kelsin para Celsius e adiciona coluna de unidade de medida
# df_temperatura_final = transform_data(df_temperatura)

# # Escreve os dados em formato parquet
# write_data(df_temperatura_final, DATA_PATH_ROOT, 2026)

In [ ]:
from datetime import datetime 

years_process = [2003, 2004, 2005, 2006, 2007, 2008, 2009
                ,2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

# years_process = [2000, 2001, 2002]

for year in years_process:
    start = datetime(2026, 7, 29).now()
    print("Start download - year : ",year, " - ", start, end="" )

    # Faz download do arquivo de temperaturas do portal Copernicus
    # retorno em formato .nc -> NetCDF (Network Common Data Form)
    ret_download         = get_t2m(year)

    # # Converte os dados de temperatura para um Spark Dataframe
    # df_temperatura       = convert_t2m_dataset_to_spark_dataframe(PROJECT_PATH, ret_download)

    # # Converte a temperatura de Kelsin para Celsius e adiciona coluna de unidade de medida
    # df_temperatura_final = transform_data(df_temperatura)

    # # Escreve os dados em formato parquet
    # write_data(df_temperatura_final, DATA_PATH_ROOT, year)

    # remove_aux_file(ret_download)

    finish = datetime(2026, 7, 29).now()

    print(f" - Download completed: {ret_download} - {finish} - {(finish - start)} \n")



Start download - year :  2003  -  2026-08-03 18:14:07.135015

2026-08-03 18:14:09,938 INFO Request ID is e49d7dab-fe96-4a90-8fdb-7825423d24d5
2026-08-03 18:14:10,144 INFO status has been updated to accepted
Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/retrieve/v1/jobs/e49d7dab-fe96-4a90-8fdb-7825423d24d5?log=True&request=True (Caused by NameResolutionError("HTTPSConnection(host='cds.climate.copernicus.eu', port=443): Failed to resolve 'cds.climate.copernicus.eu' ([Errno 11001] getaddrinfo failed)"))], attempt 1 of 500
Retrying in 120 seconds
2026-08-03 19:04:46,851 INFO status has been updated to successful


 - Download completed: 5cb734d5ed0b39511f5d862df832aac7.nc - 2026-08-03 19:05:05.476046 - 0:50:58.341031 

Start download - year :  2004  -  2026-08-03 19:05:05.476046

2026-08-03 19:05:08,595 INFO Request ID is 07528cf8-9461-4816-87f2-de3574bd9da1
2026-08-03 19:05:08,888 INFO status has been updated to accepted
2026-08-03 20:57:02,557 INFO status has been updated to running
2026-08-03 21:09:10,585 INFO status has been updated to successful


 - Download completed: 3fe04d4d84c433a2034f5ce6e36db99b.nc - 2026-08-03 21:09:28.964669 - 2:04:23.488623 

Start download - year :  2005  -  2026-08-03 21:09:28.966668

...

2026-08-03 23:47:58,203 INFO Request ID is 771285a1-6d56-453a-91ca-fb3dcbf7839f
2026-08-03 23:47:58,393 INFO status has been updated to accepted
2026-08-04 02:20:04,987 INFO status has been updated to running
2026-08-04 02:32:14,699 INFO status has been updated to successful


 - Download completed: d75d79500a12ee0f4beed2c50d76ad73.nc - 2026-08-04 02:32:34.279927 - 2:44:37.852145 

Start download - year :  2007  -  2026-08-04 02:32:34.281927

2026-08-04 02:32:36,370 INFO Request ID is e9d7b785-ef2f-412f-9386-1de39153138d
2026-08-04 02:32:36,586 INFO status has been updated to accepted
2026-08-04 05:51:29,078 INFO status has been updated to running
2026-08-04 06:05:38,987 INFO status has been updated to successful


 - Download completed: 66dbc4a222e4de351f6a9330962aa21e.nc - 2026-08-04 06:05:56.045359 - 3:33:21.763432 

Start download - year :  2008  -  2026-08-04 06:05:56.049356

2026-08-04 06:05:58,151 INFO Request ID is 6f0fe376-d476-42b4-95d0-469eaa9e80d6
2026-08-04 06:05:58,326 INFO status has been updated to accepted


Converte os dados baixados do ERA5, que estão em formato NetCDF, para um dataset (xarray.core.dataset.Dataset)

In [ ]:
ret_download = "b80f05d1220d008d67fbe12abee7958d.nc" # dados de 2023
# ret_download = "d03a86fd89e8c00cdf3bea8d8fb499f.nc" # dados de 2024
# ret_download = "507e00f8fa0894949bc892e52ab7b3b7.nc" # dados de 2025
path = f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{ret_download}"
print(path)

Renomeia nome de colunas e converte o valor da Temperatura recebida do ERA5 está em Kelvin, para converter para Celsius, subtrair 273.15

In [ ]:
df_temperatura.filter("latitude = -34.0 and longitude = -67.0").orderBy("t2m").show(10,False)

In [ ]:
drop_cols = ["valid_time", "t2m", "number"]

df_temperatura_final = \
    (df_temperatura
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("temperatura") 
                     ,"valor": (F.col("t2m") - F.lit(273.15)).cast("double")
                     ,"unidade_medida": F.lit("celsius")})
         .drop(*drop_cols)
    )

df_temperatura_final = \
    (df_temperatura_final
        .select("data_medicao"
               ,"latitude"
               ,"longitude"
               ,"indicador"
               ,"valor"
               ,"unidade_medida"))

df_temperatura_final.printSchema()

df_temperatura_final.show()

In [ ]:
# df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)

df_temperatura_final.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\mais_einstein\\dados\\ERA5-temperaturas\\2025\\ERA5_temperatura.parquet")
